# Oracle Freshness Cache Mode

This notebook demonstrates `retrieval_mode="freshness_cache"`.

What you should see:

1. Run 1 has no fresh Oracle cache row, so it calls Tavily and persists results.
2. Run 2 uses the fresh Oracle cache and returns local results.
3. The final cell deletes this notebook's demo rows so the first run stays meaningful next time.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [1]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

Loaded environment from /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/.env
Keeping proxy variables because TAVILY_USE_ENV_PROXY=1
Connected to Oracle. Table: TAVILY_DOCUMENTS
Table exists. Schema is ready.
Demo helpers are ready. Scores are ranking signals, not probabilities.
Using helper: /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/examples/oracle/demo_helpers.py


## Choose the demo query

Edit only `CACHE_QUERY` below when you want to try your own cache demo query.

In [2]:
CACHE_QUERY = "Oracle Database vector search freshness cache for Tavily results"

## Start from a clean query-specific state

In [3]:
delete_rows_for_query(CACHE_QUERY)

Deleted 0 old demo rows for this query.


0

## Run the same query twice

The low threshold is intentional for the demo. It makes the second run visibly hit Oracle cache with the deterministic demo embeddings.

In [4]:
cache_client = make_client(
    "freshness_cache",
    persistence_depth="cache_only",
    cache_ttl_seconds=3600,
    cache_score_threshold=-1.0,
)

first_results = cache_client.search(
    CACHE_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 1: cache miss, Tavily fallback", first_results)
show_persisted_rows(CACHE_QUERY, "Oracle rows after Run 1")

second_results = cache_client.search(
    CACHE_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 2: fresh Oracle cache hit", second_results)
show_persisted_rows(CACHE_QUERY, "Oracle rows after Run 2")

### Run 1: cache miss, Tavily fallback
`total=3` `origins={'foreign': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | foreign | 0.3329 | # Oracle Announces General Availability of AI Vector Search in Oracle Database 23ai. Oracle AI Vector Search is a... |
| 2 | foreign | 0.3133 | # Oracle AI Vector Search. Native AI vector search capabilities can also help large language models (LLMs) deliver... |
| 3 | foreign | 0.2969 | ... Oracle Database 23c called “AI Vector Search.” It includes vectors as a native data type, as well as vector... |

### Oracle rows after Run 1

| memory_scope | retrieval_mode | cache_hit | query_count | source_title | preview |
| --- | --- | --- | --- | --- | --- |
| cache_only | freshness_cache | 0 | 1 | Oracle AI Vector Search | # Oracle AI Vector Search. Native AI vector search capabilities can also help large... |
| cache_only | freshness_cache | 0 | 1 | Fast and Precise Business and... | ... Oracle Database 23c called “AI Vector Search.” It includes vectors as a native data... |
| cache_only | freshness_cache | 0 | 1 | Oracle Announces General Availability... | # Oracle Announces General Availability of AI Vector Search in Oracle Database 23ai.... |

### Run 2: fresh Oracle cache hit
`total=3` `origins={'local': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | local | 0.4714 | # Oracle AI Vector Search. Native AI vector search capabilities can also help large language models (LLMs) deliver... |
| 2 | local | 0.3830 | # Oracle Announces General Availability of AI Vector Search in Oracle Database 23ai. Oracle AI Vector Search is a... |
| 3 | local | 0.3785 | ... Oracle Database 23c called “AI Vector Search.” It includes vectors as a native data type, as well as vector... |

### Oracle rows after Run 2

| memory_scope | retrieval_mode | cache_hit | query_count | source_title | preview |
| --- | --- | --- | --- | --- | --- |
| cache_only | freshness_cache | 0 | 1 | Oracle AI Vector Search | # Oracle AI Vector Search. Native AI vector search capabilities can also help large... |
| cache_only | freshness_cache | 0 | 1 | Fast and Precise Business and... | ... Oracle Database 23c called “AI Vector Search.” It includes vectors as a native data... |
| cache_only | freshness_cache | 0 | 1 | Oracle Announces General Availability... | # Oracle Announces General Availability of AI Vector Search in Oracle Database 23ai.... |

## Cleanup

Run this at the end. It deletes this notebook's demo rows so the next full run starts with a cache miss again.

In [5]:
delete_rows_for_query(CACHE_QUERY)
print("Cleaned freshness-cache demo rows. Re-run from the top to see Run 1 call Tavily again.")

Deleted 3 old demo rows for this query.
Cleaned freshness-cache demo rows. Re-run from the top to see Run 1 call Tavily again.
